In [1]:
import polars as pl
from pathlib import Path
from datetime import datetime

DATA_DIR = Path('../data/processed')

rawSample = pl.read_parquet(DATA_DIR / 'raw_sample.parquet')
adFeature = pl.read_parquet(DATA_DIR / 'ad_feature.parquet')
userProfile = pl.read_parquet(DATA_DIR / 'user_profile.parquet')

behaviorLog = pl.scan_parquet(DATA_DIR / 'behavior_log.parquet')

# drop & rename
rawSample = rawSample.drop('nonclk')
userProfile = userProfile.rename({
    'new_user_class_level ': 'new_user_class_level',
    'userid': 'user',
})

In [2]:
# delete the outlier price data in adFeature
priceCap = 500_000

# delete from adFeature
adFeatureBefore = adFeature.height
adFeature = adFeature.filter(pl.col('price') <= priceCap)
print(f'adFeature: {adFeatureBefore:,} → {adFeature.height:,}  (-{adFeatureBefore - adFeature.height:,})')

# delete correspond adgroup_id from rawSample
rawSampleBefore = rawSample.height
rawSample = rawSample.filter(
    pl.col('adgroup_id').is_in(adFeature['adgroup_id'].implode())
)
print(f'rawSample: {rawSampleBefore:,} → {rawSample.height:,}  (-{rawSampleBefore - rawSample.height:,})')

# validation
print('\nFinal price stats:')
adFeature.select(
    pl.col('price').max().alias('max'),
    pl.col('price').mean().alias('mean'),
    pl.col('price').median().alias('median'),
)

adFeature: 846,811 → 846,730  (-81)
rawSample: 26,557,961 → 26,556,868  (-1,093)

Final price stats:


max,mean,median
f64,f64,f64
500000.0,658.67644,139.0


In [3]:
from datetime import datetime

# use the latest datetime of rawSample
rawSampleMax = rawSample.select(pl.col('time_stamp').max()).item()
validStart = int(datetime(2017, 4, 1).timestamp())

behaviorLog = behaviorLog.filter(
    (pl.col('time_stamp') >= validStart) &
    (pl.col('time_stamp') <= rawSampleMax)
)

print(f'Filter applied (lazy).')
print(f'  start: {datetime.fromtimestamp(validStart)}')
print(f'  end:   {datetime.fromtimestamp(rawSampleMax)}')

Filter applied (lazy).
  start: 2017-04-01 00:00:00
  end:   2017-05-13 11:59:46


In [4]:
# adFeature: delete brand
adFeature = adFeature.drop('brand')

# userProfile: null filled with zero
userProfile = userProfile.with_columns(
    pl.col('pvalue_level').fill_null(0),
    pl.col('new_user_class_level').fill_null(0),
)

# validation
print('After Step 3:')
print(f'  adFeature columns: {adFeature.columns}')
for df, name in [(adFeature, 'adFeature'), (userProfile, 'userProfile')]:
    totalNull = sum(df.select(pl.col(c).is_null().sum()).item() for c in df.columns)
    print(f'  {name}: {totalNull} nulls remaining')

After Step 3:
  adFeature columns: ['adgroup_id', 'cate_id', 'campaign_id', 'customer', 'price']
  adFeature: 0 nulls remaining
  userProfile: 0 nulls remaining


In [5]:
# Left Join
trainData = (
    rawSample
    .join(adFeature,   on='adgroup_id', how='left')
    .join(userProfile, on='user',       how='left')
)

print(f'After join shape: {trainData.shape}')
print(f'列: {trainData.columns}')

# the null distribution after join
print('\nAfter join null distribution:')
for col in trainData.columns:
    nNull = trainData.select(pl.col(col).is_null().sum()).item()
    if nNull > 0:
        pct = 100 * nNull / trainData.height
        print(f'  {col:25s}: {nNull:>10,}  ({pct:.1f}%)')

After join shape: (26556868, 17)
列: ['user', 'time_stamp', 'adgroup_id', 'pid', 'clk', 'cate_id', 'campaign_id', 'customer', 'price', 'cms_segid', 'cms_group_id', 'final_gender_code', 'age_level', 'pvalue_level', 'shopping_level', 'occupation', 'new_user_class_level']

After join null distribution:
  cms_segid                :  1,528,492  (5.8%)
  cms_group_id             :  1,528,492  (5.8%)
  final_gender_code        :  1,528,492  (5.8%)
  age_level                :  1,528,492  (5.8%)
  pvalue_level             :  1,528,492  (5.8%)
  shopping_level           :  1,528,492  (5.8%)
  occupation               :  1,528,492  (5.8%)
  new_user_class_level     :  1,528,492  (5.8%)


In [6]:
# 1. add missing flag
trainDataA = trainData.with_columns(
    has_profile=pl.col('cms_segid').is_not_null().cast(pl.Int8)
)

# 2. fill null
profileCols = [
    'cms_segid', 'cms_group_id', 'final_gender_code', 'age_level',
    'pvalue_level', 'shopping_level', 'occupation', 'new_user_class_level'
]
trainDataA = trainDataA.with_columns([pl.col(c).fill_null(0) for c in profileCols])

# 3. validation
print(f'trainDataA shape: {trainDataA.shape}')
totalNull = sum(trainDataA.select(pl.col(c).is_null().sum()).item() for c in trainDataA.columns)
print(f'nulls: {totalNull}')
print(f'has_profile distribution:')
print(trainDataA.group_by('has_profile').agg(pl.len()).sort('has_profile'))

trainDataA shape: (26556868, 18)
nulls: 0
has_profile distribution:
shape: (2, 2)
┌─────────────┬──────────┐
│ has_profile ┆ len      │
│ ---         ┆ ---      │
│ i8          ┆ u32      │
╞═════════════╪══════════╡
│ 0           ┆ 1528492  │
│ 1           ┆ 25028376 │
└─────────────┴──────────┘


In [7]:
# transfer to BTC
beijingTime = (
    pl.from_epoch('time_stamp', time_unit='s')
    .dt.replace_time_zone('UTC')
    .dt.convert_time_zone('Asia/Shanghai')
)

trainDataA = trainDataA.with_columns(
    hour=beijingTime.dt.hour().cast(pl.Int8),
    day_of_week=beijingTime.dt.weekday().cast(pl.Int8),
)

# validation
print('hour distribution(BTC):')
print(trainDataA.group_by('hour').agg(pl.len()).sort('hour'))


hour distribution(BTC):
shape: (24, 2)
┌──────┬─────────┐
│ hour ┆ len     │
│ ---  ┆ ---     │
│ i8   ┆ u32     │
╞══════╪═════════╡
│ 0    ┆ 795446  │
│ 1    ┆ 392922  │
│ 2    ┆ 212591  │
│ 3    ┆ 142636  │
│ 4    ┆ 134605  │
│ …    ┆ …       │
│ 19   ┆ 1309452 │
│ 20   ┆ 1571262 │
│ 21   ┆ 1881867 │
│ 22   ┆ 1935254 │
│ 23   ┆ 1406997 │
└──────┴─────────┘


In [14]:
encodeCols = ['pid', 'cate_id', 'user', 'adgroup_id', 'campaign_id', 'customer']

trainDataA = trainDataA.with_columns([
    (pl.col(c).rank('dense') - 1).cast(pl.UInt32).alias(c)
    for c in encodeCols
])

# validation
print(f"{'col':<15} {'min':>8} {'max':>10} {'n_unique':>10}")
print('-' * 50)
for col in encodeCols:
    s = trainDataA[col]
    print(f"{col:<15} {s.min():>8,} {s.max():>10,} {s.n_unique():>10,}")

print(f'\nFinal shape: {trainDataA.shape}')

col                  min        max   n_unique
--------------------------------------------------
pid                    0          1          2
cate_id                0      6,768      6,769
user                   0  1,141,725  1,141,726
adgroup_id             0    846,729    846,730
campaign_id            0    423,419    423,420
customer               0    255,868    255,869

Final shape: (26556868, 20)


In [ ]:
# add feature log_price
trainDataA = trainDataA.with_columns(
    log_price=(pl.col('price') + 1).log()
)

# validation
print(trainDataA.select(
    pl.col('price').min().alias('price_min'),
    pl.col('price').max().alias('price_max'),
    pl.col('log_price').min().alias('logp_min'),
    pl.col('log_price').max().alias('logp_max'),
    pl.col('log_price').mean().alias('logp_mean'),
))

shape: (1, 5)
┌───────────┬───────────┬──────────┬───────────┬───────────┐
│ price_min ┆ price_max ┆ logp_min ┆ logp_max  ┆ logp_mean │
│ ---       ┆ ---       ┆ ---      ┆ ---       ┆ ---       │
│ f64       ┆ f64       ┆ f64      ┆ f64       ┆ f64       │
╞═══════════╪═══════════╪══════════╪═══════════╪═══════════╡
│ 0.01      ┆ 500000.0  ┆ 0.00995  ┆ 13.122365 ┆ 5.101672  │
└───────────┴───────────┴──────────┴───────────┴───────────┘


In [17]:
from datetime import datetime, timezone

# Split Point
valTime  = int(datetime(2017, 5, 12, 0, 0, 0, tzinfo=timezone.utc).timestamp())
testTime = int(datetime(2017, 5, 13, 0, 0, 0, tzinfo=timezone.utc).timestamp())

trainA = trainDataA.filter(pl.col('time_stamp') < valTime)
valA   = trainDataA.filter(
    (pl.col('time_stamp') >= valTime) & (pl.col('time_stamp') < testTime)
)
testA  = trainDataA.filter(pl.col('time_stamp') >= testTime)

# validation
print(f'Train: {trainA.height:>11,} rows  ({trainA.height/trainDataA.height*100:.1f}%)')
print(f'Val:   {valA.height:>11,} rows  ({valA.height/trainDataA.height*100:.1f}%)')
print(f'Test:  {testA.height:>11,} rows  ({testA.height/trainDataA.height*100:.1f}%)')
print(f'Sum:   {trainA.height + valA.height + testA.height:>11,}')

# CTR sanity check
print(f'\nTrain CTR: {trainA["clk"].mean():.4f}')
print(f'Val   CTR: {valA["clk"].mean():.4f}')
print(f'Test  CTR: {testA["clk"].mean():.4f}')

# Time series check
print(f'\nTrain end: {datetime.fromtimestamp(trainA["time_stamp"].max(), tz=timezone.utc)}')
print(f'Val  start: {datetime.fromtimestamp(valA["time_stamp"].min(), tz=timezone.utc)}')
print(f'Val  end:   {datetime.fromtimestamp(valA["time_stamp"].max(), tz=timezone.utc)}')
print(f'Test start: {datetime.fromtimestamp(testA["time_stamp"].min(), tz=timezone.utc)}')


Train:  20,437,223 rows  (77.0%)
Val:     3,271,268 rows  (12.3%)
Test:    2,848,377 rows  (10.7%)
Sum:    26,556,868

Train CTR: 0.0519
Val   CTR: 0.0493
Test  CTR: 0.0504

Train end: 2017-05-11 23:59:59+00:00
Val  start: 2017-05-12 00:00:00+00:00
Val  end:   2017-05-12 23:59:59+00:00
Test start: 2017-05-13 00:00:00+00:00


In [18]:
trainB = trainA.filter(pl.col('has_profile') == 1).drop('has_profile')
valB   = valA.filter(pl.col('has_profile') == 1).drop('has_profile')
testB  = testA.filter(pl.col('has_profile') == 1).drop('has_profile')

# validation
print(f'{"set":<8} {"Version A":>15} {"Version B":>15} {"diff":>15}')
print('-' * 60)
for name, a, b in [('Train', trainA, trainB), ('Val', valA, valB), ('Test', testA, testB)]:
    diff = a.height - b.height
    print(f'{name:<8} {a.height:>15,} {b.height:>15,} {diff:>15,}')

# CTR comparison
print(f'\n{"set":<8} {"A CTR":>10} {"B CTR":>10}')
print('-' * 35)
for name, a, b in [('Train', trainA, trainB), ('Val', valA, valB), ('Test', testA, testB)]:
    print(f'{name:<8} {a["clk"].mean():>10.4f} {b["clk"].mean():>10.4f}')

# B should have 1 column less (drop has_profile)
print(f'\nA cols: {trainA.width}')
print(f'B cols: {trainB.width}')

set            Version A       Version B            diff
------------------------------------------------------------
Train         20,437,223      19,281,955       1,155,268
Val            3,271,268       3,078,536         192,732
Test           2,848,377       2,667,885         180,492

set           A CTR      B CTR
-----------------------------------
Train        0.0519     0.0518
Val          0.0493     0.0492
Test         0.0504     0.0502

A cols: 21
B cols: 20


In [ ]:
# version A
GLOBAL_CTR_A = trainA['clk'].mean()
print(f'Version A 全局 CTR: {GLOBAL_CTR_A:.4f}')

userAggA = trainA.group_by('user').agg(
    user_imp_count=pl.len(),
    user_ctr=pl.col('clk').mean(),
)
adAggA = trainA.group_by('adgroup_id').agg(
    ad_imp_count=pl.len(),
    ad_ctr=pl.col('clk').mean(),
)
cateAggA = trainA.group_by('cate_id').agg(
    cate_ctr=pl.col('clk').mean(),
)

def addAggregates(df, userAgg, adAgg, cateAgg, globalCtr):
    return (
        df
        .join(userAgg, on='user', how='left')
        .join(adAgg, on='adgroup_id', how='left')
        .join(cateAgg, on='cate_id', how='left')
        .with_columns([
            # train 没见过的 user/ad/cate → 填全局均值或 0
            pl.col('user_imp_count').fill_null(0),
            pl.col('user_ctr').fill_null(globalCtr),
            pl.col('ad_imp_count').fill_null(0),
            pl.col('ad_ctr').fill_null(globalCtr),
            pl.col('cate_ctr').fill_null(globalCtr),
        ])
    )

trainA = addAggregates(trainA, userAggA, adAggA, cateAggA, GLOBAL_CTR_A)
valA   = addAggregates(valA,   userAggA, adAggA, cateAggA, GLOBAL_CTR_A)
testA  = addAggregates(testA,  userAggA, adAggA, cateAggA, GLOBAL_CTR_A)

print(f'\nversion A shape:')
print(f'  trainA: {trainA.shape}')
print(f'  valA:   {valA.shape}')
print(f'  testA:  {testA.shape}')

Version A 全局 CTR: 0.0519

A 版本 shape:
  trainA: (20437223, 26)
  valA:   (3271268, 26)
  testA:  (2848377, 26)


In [20]:
GLOBAL_CTR_B = trainB['clk'].mean()
print(f'Version B 全局 CTR: {GLOBAL_CTR_B:.4f}')

userAggB = trainB.group_by('user').agg(
    user_imp_count=pl.len(),
    user_ctr=pl.col('clk').mean(),
)
adAggB = trainB.group_by('adgroup_id').agg(
    ad_imp_count=pl.len(),
    ad_ctr=pl.col('clk').mean(),
)
cateAggB = trainB.group_by('cate_id').agg(
    cate_ctr=pl.col('clk').mean(),
)

trainB = addAggregates(trainB, userAggB, adAggB, cateAggB, GLOBAL_CTR_B)
valB   = addAggregates(valB,   userAggB, adAggB, cateAggB, GLOBAL_CTR_B)
testB  = addAggregates(testB,  userAggB, adAggB, cateAggB, GLOBAL_CTR_B)

print(f'\nversion B shape:')
print(f'  trainB: {trainB.shape}')
print(f'  valB:   {valB.shape}')
print(f'  testB:  {testB.shape}')


Version B 全局 CTR: 0.0518

version B shape:
  trainB: (19281955, 25)
  valB:   (3078536, 25)
  testB:  (2667885, 25)


In [21]:
from pathlib import Path

SAVE_DIR = Path('../data/processed/wide')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    'trainA': trainA, 'valA': valA, 'testA': testA,
    'trainB': trainB, 'valB': valB, 'testB': testB,
}

for name, df in datasets.items():
    path = SAVE_DIR / f'{name}.parquet'
    df.write_parquet(path, compression='snappy')
    sizeMb = path.stat().st_size / (1024**2)
    print(f'{name:8s} → {df.height:>11,} rows, {df.width} cols, {sizeMb:.1f} MB')

total = sum(p.stat().st_size for p in SAVE_DIR.glob('*.parquet')) / (1024**3)
print(f'\nTotal disk: {total:.2f} GB')

trainA   →  20,437,223 rows, 26 cols, 458.1 MB
valA     →   3,271,268 rows, 26 cols, 80.3 MB
testA    →   2,848,377 rows, 26 cols, 68.1 MB
trainB   →  19,281,955 rows, 25 cols, 429.9 MB
valB     →   3,078,536 rows, 25 cols, 75.5 MB
testB    →   2,667,885 rows, 25 cols, 63.8 MB

Total disk: 1.15 GB
